# 02 · Data preparation: from raw telemetry to model inputs

This notebook turns the raw files into the inputs of the models, using only **pandas and numpy**:

| Output (in `data/processed/`) | Used by | One row per |
|---|---|---|
| `service_features.parquet`: hand-made features | rules (03), XGBoost (04) | (case, candidate service) |
| `cnn_inputs.npy` + `cnn_rows.parquet`: scaled time series | 1D CNN (05) | (case, service) |

Every step is first shown on **one example case** (the CPU fault on `checkoutservice` from notebook 01), then the same functions run on all 270 cases.

### The features: "how differently did this service behave after the fault?"
Each feature compares the 12 minutes *after* the injection with the 12 minutes *before*, using only that case's own data:

| Source | Features (per service) | Idea |
|---|---|---|
| Metrics | for each metric type (cpu, mem, diskio, socket, workload, latency-50, latency-90, error): **max robust z-score** after the fault, mean z-score, relative change of the mean | "how far outside its normal range did it go?" |
| Logs | change in log volume, change in error-like lines, share of log *templates* never seen before the fault | "did it start complaining, or go quiet?" |
| Traces | change in span count, p50/p95 latency change, worst single operation's p95 change, error-rate change | "did its requests get slower or fail?" |
| Ranks | every feature's **rank among the services of the same case** | makes systems with 12 and 68 services comparable |

### Robust z-score (the key statistic)
For a metric, take the normal window, compute its **median** and **IQR** (spread of the middle 50%). After the fault, `z = (value − median) / IQR`. Median/IQR instead of mean/std because one spike in the normal window would otherwise inflate the "normal" spread. This is the same idea as **BARO** (FSE 2024), a strong published method on this benchmark.

### Leakage rules
- No feature uses the label, other cases, or anything after the case's own time window.
- **Service names are not features.** Otherwise the model would just learn "the benchmark only breaks these 5 services", a shortcut that is useless in real life.
- Features are computed per case with fixed rules (nothing is *fitted*), so they cannot leak test information into training.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
ROOT = Path.cwd().parent                                  # the project folder (notebooks run inside notebooks/)
RAW, PROCESSED, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports" / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

METRIC_TYPES = ["cpu", "mem", "diskio", "socket", "workload", "latency-50", "latency-90", "error"]

cases = pd.read_parquet(RAW / "cases.parquet")
cases = cases[cases["suite"] == "RE2"].sort_values("case").reset_index(drop=True)
cases["pair"] = cases["system"] + "|" + cases["root_cause_service"] + "|" + cases["fault"]   # the 3 repetitions share a pair

# trace features only if traces were downloaded for every Online Boutique and Train Ticket case (90 + 90)
USE_TRACES = len(list(RAW.glob("*/traces.parquet"))) == 180
example = cases[cases["case"] == "re2ob_checkoutservice_cpu_1"].iloc[0]
print(len(cases), "cases | trace features:", USE_TRACES)

# Part A · Features for the rules and XGBoost

## Step 1 · Metrics in long format
As in notebook 01: `melt` the wide metrics file into one row per (second, service, metric), so every per-service calculation becomes a `groupby(["service", "metric"])`.

In [ ]:
def load_metrics(case):
    wide = pd.read_parquet(RAW / case / "metrics.parquet")
    long = wide.melt(id_vars="time", var_name="column", value_name="value")
    long[["service", "metric"]] = long["column"].str.rsplit("_", n=1, expand=True)
    long = long[long["metric"].isin(METRIC_TYPES)]
    return long.drop(columns="column")

long = load_metrics(example["case"])
print(long.shape)
long.head()

## Step 2 · What is "normal" for each metric?
Split at the injection time: `before` = the 12 normal minutes, `after` = the 12 minutes with the fault. For every (service, metric) the normal window is described by:
- **median**: the typical value,
- **IQR** (75th − 25th percentile): the typical spread, not affected by a few spikes.

If the IQR is 0 (e.g. an error counter that is always 0) we use the standard deviation instead, and if that is 0 too, a tiny number, so we never divide by zero. "0" means below 1e-9: for a perfectly flat series the computer returns rounding noise like 3e-17 instead of an exact 0.

In [ ]:
def normal_behaviour(before):
    g = before.groupby(["service", "metric"], sort=False)["value"]       # sort=False: keep the file's order
    normal = pd.DataFrame({"n_before": g.count(), "median": g.median(),
                           "iqr": g.quantile(0.75) - g.quantile(0.25),
                           "std": g.std(ddof=0), "mean_before": g.mean()})
    tiny = 1e-3 * np.maximum(normal["median"].abs(), 1e-3)
    normal["spread"] = normal["iqr"].where(normal["iqr"] > 0, normal["std"])   # IQR, else std ...
    normal["spread"] = normal["spread"].where(normal["spread"] > 1e-9, tiny)  # ... else a tiny number
    return normal

values = long.dropna(subset=["value"])
normal = normal_behaviour(values[values["time"] < example["inject_time"]])
normal.loc[example["root_cause_service"]].round(3)

## Step 3 · Metric features
For every second after the fault: `z = (value − median) / spread`, i.e. "how many normal spreads away from normal" (capped at ±10,000). Then per (service, metric):

| Feature | Formula | Meaning |
|---|---|---|
| `maxz` | log(1 + max \|z\|) | the most extreme moment |
| `meanz` | signed log of the mean z | a sustained shift, with its direction (up / down) |
| `rel` | signed log of (mean after − mean before) / \|mean before\| | relative change of the average |

**Signed log** = sign(x) · log(1 + \|x\|): keeps the direction, but turns 10,000 into 9.2, so one flat metric that suddenly moves can't dominate. A (service, metric) needs at least 10 values before *and* after.

Finally `unstack` turns the long result into **one row per service**: 8 metric types × 3 features = 24 columns, plus `m_all_maxz` (its most extreme metric) and `m_n_metrics` (how many metric types it has).

In [ ]:
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

METRIC_COLS = [f"m_{t.replace('-', '')}_{s}" for t in METRIC_TYPES for s in ["maxz", "meanz", "rel"]]   # m_cpu_maxz, m_cpu_meanz, ...
MAXZ_COLS = [c for c in METRIC_COLS if c.endswith("_maxz")]

def metric_features(case, inject_time):
    values = load_metrics(case).dropna(subset=["value"])
    before = values[values["time"] < inject_time]
    after = values[values["time"] >= inject_time]
    normal = normal_behaviour(before)

    # z-score of every value after the fault, using the normal median / spread of its (service, metric)
    after = after.merge(normal[["median", "spread"]], left_on=["service", "metric"], right_index=True)
    after["z"] = ((after["value"] - after["median"]) / after["spread"]).clip(-1e4, 1e4)
    after["abs_z"] = after["z"].abs()
    g = after.groupby(["service", "metric"], sort=False)
    stats = normal.join(pd.DataFrame({"n_after": g.size(), "max_abs_z": g["abs_z"].max(),
                                      "mean_z": g["z"].mean(), "mean_after": g["value"].mean()}), how="inner")
    stats = stats[(stats["n_before"] >= 10) & (stats["n_after"] >= 10)]

    stats["maxz"] = np.log1p(stats["max_abs_z"])
    stats["meanz"] = signed_log(stats["mean_z"])
    stats["rel"] = signed_log((stats["mean_after"] - stats["mean_before"]) / (stats["mean_before"].abs() + 1e-9))

    # long -> wide: one row per service, one column per (metric, feature)
    wide = stats[["maxz", "meanz", "rel"]].unstack("metric")
    wide = wide.reindex(stats.index.get_level_values("service").unique())          # keep the services in the file's order
    wide.columns = [f"m_{metric.replace('-', '')}_{feature}" for feature, metric in wide.columns]   # ('maxz', 'cpu') -> 'm_cpu_maxz'
    wide = wide.reindex(columns=METRIC_COLS)
    wide["m_all_maxz"] = wide[MAXZ_COLS].max(axis=1)
    wide["m_n_metrics"] = wide[MAXZ_COLS].notna().sum(axis=1)
    wide.index.name = "service"
    return wide

m = metric_features(example["case"], example["inject_time"])
print(m.shape, "(services, metric features)")
m.sort_values("m_all_maxz", ascending=False).head(5).round(2)

## Step 4 · Log features
Per service, the 12 minutes after the fault against the 12 minutes before:

| Feature | Meaning |
|---|---|
| `l_rate_ratio` | log of (lines per second after / before): did it start talking a lot, or go quiet? |
| `l_err_ratio` | the same for **error-like lines** (*error, exception, fail, timeout, refused, unavailable, denied* or a 5xx code) |
| `l_err_share_post` | share of its lines after the fault that are error-like |
| `l_new_tmpl_share` | share of its lines after the fault whose **template** never appeared before |

A *template* is the message with numbers and long hex IDs replaced by `#`, so `took 35 ms` and `took 812 ms` count as the same message. The small constants (0.1, 0.01) avoid log(0) for services that log nothing.

Service names differ between sources (`frontend` in metrics, `frontendservice` in traces), so they are **normalised** before matching: lower case, letters and digits only, the word "service" removed.

In [ ]:
def normalise(names):
    return names.str.lower().str.replace(r"[^a-z0-9]", "", regex=True).str.replace("service", "")

# \b = whole word; (?: ) only groups the alternatives; 5\d\d = 500-599
ERROR_PATTERN = r"\b(?:error|exception|fail|failed|failure|timeout|timed out|refused|unavailable|denied|5\d\d)\b"
LOG_COLS = ["l_rate_ratio", "l_err_ratio", "l_err_share_post", "l_new_tmpl_share"]

def log_features(row, services):
    path = RAW / row["case"] / "logs.parquet"
    if not path.exists():                                     # one Train Ticket case has no logs
        return pd.DataFrame(index=services)
    logs = pd.read_parquet(path)
    logs["key"] = normalise(logs["container_name"])
    logs["is_after"] = logs["timestamp"] >= row["inject_time"]
    logs["is_error"] = logs["message"].str.contains(ERROR_PATTERN, case=False, regex=True)
    logs["template"] = logs["message"].str.replace(r"[0-9a-fA-F]{8,}|\d+", "#", regex=True).str.slice(0, 120)

    before, after = logs[~logs["is_after"]], logs[logs["is_after"]]
    seen = before[["key", "template"]].drop_duplicates().assign(seen_before=True)   # templates each service used before
    after = after.merge(seen, on=["key", "template"], how="left")
    after["new_template"] = after["seen_before"].isna()

    f = pd.DataFrame({"lines_before": before.groupby("key").size(),
                      "lines_after": after.groupby("key").size(),
                      "errors_before": before.groupby("key")["is_error"].sum(),
                      "errors_after": after.groupby("key")["is_error"].sum(),
                      "new_after": after.groupby("key")["new_template"].sum()}).fillna(0)
    sec_before = max(row["inject_time"] - row["time_start"], 1)
    sec_after = max(row["time_end"] - row["inject_time"] + 1, 1)
    f["l_rate_ratio"] = np.log((f["lines_after"] / sec_after + 0.1) / (f["lines_before"] / sec_before + 0.1))
    f["l_err_ratio"] = np.log((f["errors_after"] / sec_after + 0.01) / (f["errors_before"] / sec_before + 0.01))
    f["l_err_share_post"] = (f["errors_after"] / f["lines_after"]).fillna(0)
    f["l_new_tmpl_share"] = (f["new_after"] / f["lines_after"]).fillna(0)
    return f[LOG_COLS].astype(float).reindex(normalise(services)).set_axis(services)   # back to the metric service names

l = log_features(example, m.index)
l.sort_values("l_err_ratio", ascending=False).head(5).round(2)

## Step 5 · Trace features
A *span* is one request handled by one service, with its duration and status. Per service, after against before:

| Feature | Meaning |
|---|---|
| `t_count_ratio` | log of (spans after + 1) / (spans before + 1) |
| `t_p50_ratio`, `t_p95_ratio` | log ratio of the median / 95th-percentile span duration |
| `t_worst_op_p95_ratio` | the same p95 ratio for its **worst single operation** (one slow endpoint can hide in the average) |
| `t_err_rate_diff` | error rate after − error rate before |

With fewer than 20 spans before or after, percentiles are unreliable, so only the count change is kept. Sock Shop has no traces, so these features are missing there.

In [ ]:
TRACE_COLS = ["t_count_ratio", "t_p50_ratio", "t_p95_ratio", "t_worst_op_p95_ratio", "t_err_rate_diff"]

def trace_features(row, services):
    path = RAW / row["case"] / "traces.parquet"
    if not path.exists():                                     # Sock Shop, or traces not downloaded
        return pd.DataFrame(index=services)
    tr = pd.read_parquet(path, columns=["serviceName", "operationName", "startTimeMillis", "duration", "statusCode"])
    tr["key"] = normalise(tr["serviceName"])
    tr["is_after"] = tr["startTimeMillis"] / 1000 >= row["inject_time"]
    tr["is_error"] = tr["statusCode"].fillna(0) != 0
    tr["duration"] = tr["duration"].astype(float)

    before, after = tr[~tr["is_after"]].groupby("key"), tr[tr["is_after"]].groupby("key")
    f = pd.DataFrame({"spans_before": before.size(), "spans_after": after.size(),
                      "p50_before": before["duration"].median(), "p50_after": after["duration"].median(),
                      "p95_before": before["duration"].quantile(0.95), "p95_after": after["duration"].quantile(0.95),
                      "err_before": before["is_error"].mean(), "err_after": after["is_error"].mean()})
    f[["spans_before", "spans_after"]] = f[["spans_before", "spans_after"]].fillna(0)

    # p95 of every (service, operation) before and after -> the operation that slowed down most
    op = tr.groupby(["key", "operationName", "is_after"])["duration"].quantile(0.95).unstack("is_after")
    op_change = np.log((op[True] + 1) / (op[False] + 1))

    f["t_count_ratio"] = np.log((f["spans_after"] + 1) / (f["spans_before"] + 1))
    f["t_p50_ratio"] = np.log((f["p50_after"] + 1) / (f["p50_before"] + 1))
    f["t_p95_ratio"] = np.log((f["p95_after"] + 1) / (f["p95_before"] + 1))
    f["t_worst_op_p95_ratio"] = op_change.groupby("key").max()
    f["t_err_rate_diff"] = f["err_after"] - f["err_before"]
    few = (f["spans_before"] < 20) | (f["spans_after"] < 20)
    f.loc[few, ["t_p50_ratio", "t_p95_ratio", "t_worst_op_p95_ratio", "t_err_rate_diff"]] = np.nan
    return f[TRACE_COLS].astype(float).reindex(normalise(services)).set_axis(services)

t = trace_features(example, m.index)
t.sort_values("t_p95_ratio", ascending=False).head(5).round(2)

## Step 6 · All sources together, for all 270 cases
`case_features` joins the three tables: one row per candidate service (every service with valid metrics). The loop runs it on every case. Searching the log text is the slow part (Train Ticket logs are ~180 MB of text per case), so this takes roughly 15–25 minutes.

In [ ]:
def case_features(row):
    f = metric_features(row["case"], row["inject_time"])
    f = f.join(log_features(row, f.index))
    if USE_TRACES:
        f = f.join(trace_features(row, f.index))
    f["case"] = row["case"]
    return f

example_features = case_features(example)
print(example_features.shape)
example_features.loc[[example["root_cause_service"]]].round(2)

In [ ]:
parts = []
for i, row in cases.iterrows():
    parts.append(case_features(row))
    if (i + 1) % 30 == 0:
        print(f"{i + 1}/{len(cases)} cases done")
features = pd.concat(parts).reset_index()
features.shape

### Within-case ranks, then the labels
Systems have 12–68 services, so a raw value like "max z = 6" means different things in different systems. For every feature we add its **percentile rank inside its case**: `groupby("case").rank(pct=True)` of the absolute value, where 1.0 = the most extreme service of that case.

Then the labels: `is_rc = 1` for the true root-cause service. This is the only place the label is used.

In [ ]:
raw_cols = [c for c in features.columns if c.startswith(("m_", "l_", "t_")) and c != "m_n_metrics"]
ranks = features[raw_cols].abs().groupby(features["case"]).rank(pct=True)
features = features.join(ranks.add_suffix("_rank"))
features["n_services"] = features.groupby("case")["service"].transform("size")

labels = cases[["case", "system", "fault", "pair", "repetition", "root_cause_service"]]
features = features.merge(labels, on="case", how="left")
features["is_rc"] = (features["service"] == features["root_cause_service"]).astype(int)
features.insert(0, "case", features.pop("case"))                    # move "case" to the first column

# every case is there, and its root-cause service is one of its candidates
assert features["case"].nunique() == len(cases)
assert features.groupby("case")["is_rc"].sum().eq(1).all()
features.to_parquet(PROCESSED / "service_features.parquet", index=False)
print("saved", features.shape, "-> data/processed/service_features.parquet")
features.head(3)

## Look at the result
How many services have each source of evidence?

In [ ]:
raw = features.loc[:, ~features.columns.str.endswith("_rank")]
has = pd.DataFrame({"metrics": raw.filter(regex="^m_").notna().any(axis=1),
                    "logs": raw.filter(regex="^l_").notna().any(axis=1),
                    "traces": raw.filter(regex="^t_").notna().any(axis=1)})
has.groupby(features["system"]).mean().round(2).rename_axis("share of services with this source")

Not every service has every source: some services write no logs, Sock Shop has no traces, databases have no latency metrics. **XGBoost handles missing values natively**, so we keep them as missing instead of inventing values.

## Do the features separate root causes from innocent services?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, col in zip(axes, ["m_all_maxz", "m_all_maxz_rank", "l_err_ratio"]):
    sns.boxplot(data=features, x="system", y=col, hue=features["is_rc"].map({0: "other service", 1: "root cause"}), ax=ax, fliersize=1)
    ax.set_title(col); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The root cause usually has the most extreme metric change in its case (rank close to 1.0), but the boxes overlap: innocent services can look just as extreme (e.g. a caller suffering from the root cause's delay).

## Which metric reacts to which fault? (root-cause services only)

In [ ]:
rc = features[features["is_rc"] == 1]
heat = rc.groupby("fault")[MAXZ_COLS].median().rename(columns=lambda c: c[2:-5])     # 'm_cpu_maxz' -> 'cpu'
plt.figure(figsize=(9, 3.8)); sns.heatmap(heat, annot=True, fmt=".1f", cmap="Reds")
plt.title("median log(1 + max |z|) of the root-cause service, by fault type"); plt.tight_layout()
plt.savefig(FIG / "fault_signatures.png", dpi=120); plt.show()

**This is the "signature" of each fault:** CPU faults show up in `cpu`, memory faults in `mem`, disk faults in `diskio`, while **delay / loss** faults mostly show in latency and are the hard ones (the latency also rises in the services that call the culprit). These signatures are also what lets us classify the **fault type** (notebook 04).

# Part B · Input for the deep-learning model (notebook 05)
The CNN does not use the features above; it reads the **time series** itself. For every (case, service):
1. scale each metric with the same normal median / spread as in Step 2: `z = (value − median) / spread` (missing values → 0), then the signed log;
2. average into **10-second steps** over the 12 minutes before and the 12 minutes after the injection → **144 steps**, fault in the middle;
3. stack the 8 metric types as 8 *channels* (like the colour channels of an image), plus 8 **presence channels** (1 if the service has that metric type), so the network can tell "flat" from "not measured".

Result per case: an array of shape (services, 16 channels, 144 steps).

In [ ]:
STEP, HALF = 10, 720                     # 10-second steps; 720 s = 12 minutes on each side of the injection
N_STEPS = 2 * HALF // STEP               # 144

def cnn_input(case, inject_time):
    long = load_metrics(case)
    normal = normal_behaviour(long[long["time"] < inject_time].dropna(subset=["value"]))
    normal = normal[normal["n_before"] >= 10]

    long = long.merge(normal[["median", "spread"]], left_on=["service", "metric"], right_index=True, how="left")
    long["z"] = signed_log(((long["value"] - long["median"]) / long["spread"]).fillna(0).clip(-1e4, 1e4))
    long["step"] = (long["time"] - (inject_time - HALF)) // STEP
    long = long[(long["step"] >= 0) & (long["step"] < N_STEPS)]

    services = sorted(long["service"].unique())
    grid = pd.MultiIndex.from_product([services, METRIC_TYPES], names=["service", "metric"])   # every (service, metric)
    values = long.groupby(["service", "metric", "step"])["z"].mean().unstack("step")
    values = values.reindex(index=grid, columns=range(N_STEPS)).fillna(0)
    present = normal["n_before"].reindex(grid).notna().astype(float)

    values = values.to_numpy().reshape(len(services), len(METRIC_TYPES), N_STEPS)
    present = np.repeat(present.to_numpy().reshape(len(services), len(METRIC_TYPES), 1), N_STEPS, axis=2)
    return services, np.concatenate([values, present], axis=1).astype("float32")

services, X = cnn_input(example["case"], example["inject_time"])
print("array:", X.shape, "= (services, channels, time steps)")
fig, axes = plt.subplots(1, 2, figsize=(15, 3.2))
for ax, name in zip(axes, [example["root_cause_service"], "frontend"]):
    sns.heatmap(X[services.index(name), :8], ax=ax, cmap="RdBu_r", center=0, yticklabels=METRIC_TYPES, xticklabels=24)
    ax.axvline(N_STEPS // 2, color="k", ls="--")
    ax.set_title(f"{name}: what the CNN sees"); ax.set_xlabel("10-second step (fault injected at step 72)")
plt.tight_layout(); plt.savefig(FIG / "cnn_input_example.png", dpi=120); plt.show()

In [ ]:
arrays, rows = [], []
for i, row in cases.iterrows():
    services, X = cnn_input(row["case"], row["inject_time"])
    arrays.append(X)
    rows.append(pd.DataFrame({"case": row["case"], "service": services}))
    if (i + 1) % 90 == 0:
        print(f"{i + 1}/{len(cases)} cases done")

cnn_rows = pd.concat(rows, ignore_index=True).merge(cases[["case", "system", "pair", "root_cause_service"]], on="case")
cnn_rows["is_rc"] = (cnn_rows["service"] == cnn_rows["root_cause_service"]).astype(int)
cnn_X = np.concatenate(arrays)
assert len(cnn_rows) == len(cnn_X) and cnn_rows.groupby("case")["is_rc"].sum().eq(1).all()
np.save(PROCESSED / "cnn_inputs.npy", cnn_X)
cnn_rows.to_parquet(PROCESSED / "cnn_rows.parquet", index=False)
print("saved", cnn_X.shape, "-> data/processed/cnn_inputs.npy (+ cnn_rows.parquet: the case and service of each row)")

### Recap
- **XGBoost input:** one row per (case, candidate service), 26 metric + 4 log + 5 trace features, plus their within-case ranks.
- **CNN input:** per (case, service) a 16 × 144 array of scaled metric series.
- Both use robust (median / spread) statistics of each case's own normal window; no labels, no service names, nothing fitted.

➡️ Next: **03_baselines**: how far do simple rules get?